In [1]:

pip install feedparser beautifulsoup4 pandas nltk

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 1.8 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=11427c3753d340e22553bf1b35279968436034fb6ff4feb6c68c99befb8dfd44
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k


In [2]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the base directory for your project in Google Drive
base_drive_path = '/content/drive/MyDrive/'

# Define the specific folder path for daily results
target_folder = os.path.join(base_drive_path, 'MTech_VNIT_3B','Sem3', 'NLP', 'RSS', 'DailyRun')

# Create the directory if it doesn't exist
os.makedirs(target_folder, exist_ok=True)

print(f"Google Drive mounted at {base_drive_path}")
print(f"Target folder for daily results created/verified at: {target_folder}")

# You can now use 'target_folder' to save your CSV files, e.g.:
# df_market_sentiment.to_csv(os.path.join(target_folder, filename), index=False)

Mounted at /content/drive
Google Drive mounted at /content/drive/MyDrive/
Target folder for daily results created/verified at: /content/drive/MyDrive/MTech_VNIT_3B/Sem3/NLP/RSS/DailyRun


In [3]:
import feedparser
import pandas as pd
from datetime import datetime, timedelta
import time
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from bs4 import BeautifulSoup
import os # Import os module for path operations

# Download VADER lexicon (run once)
nltk.download('vader_lexicon', quiet=True)

def capture_multiple_rss_sentiment(days_back=15):
    # List of various Indian Market RSS Feeds
    # rss_urls1 = [
    #     "https://economictimes.indiatimes.com/markets/rssfeeds/1977021501.cms", # Economic Times Markets
    #     "https://www.livemint.com/rss/markets",                                 # LiveMint Markets
    #     "https://www.business-standard.com/rss/markets-101.rss"                 # Business Standard Markets
    # ]
    rss_urls = [
        "https://economictimes.indiatimes.com/markets/rssfeeds/1977021501.cms",
        "https://www.livemint.com/rss/markets",
        "https://www.business-standard.com/rss/markets-101.rss",
        "https://www.moneycontrol.com/rss/marketreports.xml",
        "https://www.thehindubusinessline.com/markets/feeder/default.rss",
        "https://www.financialexpress.com/market/feed/",
        #"https://www.nseindia.com/static/rss-feed",
        "https://indianexpress.com/section/business/feed/"
    ]



    data = []
    date_limit = datetime.now() - timedelta(days=days_back)
    sia = SentimentIntensityAnalyzer()

    for url in rss_urls:
        print(f"Fetching from: {url}")
        feed = feedparser.parse(url)

        for entry in feed.entries:
            # 1. Extract Text & Clean HTML
            title = entry.get('title', '')
            raw_summary = entry.get('summary', '')

            # Some RSS descriptions contain raw HTML (like image tags); this strips it to pure text
            clean_summary = BeautifulSoup(raw_summary, "html.parser").get_text(separator=" ", strip=True)
            full_text = f"{title}. {clean_summary}"

            # 2. Safely Parse the Publication Date
            if 'published_parsed' in entry and entry.published_parsed:
                pub_date = datetime.fromtimestamp(time.mktime(entry.published_parsed))
            else:
                pub_date = datetime.now() # Fallback if no date is provided

            # 3. Filter for the target timeframe
            if pub_date >= date_limit:
                # 4. Analyze Sentiment
                sentiment_scores = sia.polarity_scores(full_text)
                compound_score = sentiment_scores['compound']

                # Categorize
                if compound_score >= 0.05:
                    sentiment = 'Positive'
                elif compound_score <= -0.05:
                    sentiment = 'Negative'
                else:
                    sentiment = 'Neutral'

                # 5. Store the data
                data.append({
                    'Source': url.split('/')[2].replace('www.', ''), # Extract domain name cleanly
                    'Date': pub_date.strftime("%Y-%m-%d %H:%M:%S"),
                    'Text': full_text,
                    'Sentiment': sentiment,
                    'Score': compound_score
                })

    # Convert to DataFrame
    df = pd.DataFrame(data)

    # Remove any duplicate articles based on the text to keep the dataset clean
    if not df.empty:
        df = df.drop_duplicates(subset=['Text'])
        # Sort by date, newest first
        df = df.sort_values(by='Date', ascending=False)

    return df

# Run the pipeline
df_market_sentiment = capture_multiple_rss_sentiment(days_back=15)

if df_market_sentiment is not None and not df_market_sentiment.empty:
    filename = "indian_market_RSSFeed_daily"
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{filename}_{timestamp}.csv"

    # Construct the full path using target_folder
    full_filepath = os.path.join(target_folder, filename)

    # Check if the file already exists to append, or create new
    try:
        # Read from the full_filepath
        existing_df = pd.read_csv(full_filepath)
        combined_df = pd.concat([existing_df, df_market_sentiment]).drop_duplicates(subset=['Text'])
        combined_df.to_csv(full_filepath, index=False)
        print(f"\nAppended new data. Total records now: {len(combined_df)}")
    except FileNotFoundError:
        df_market_sentiment.to_csv(full_filepath, index=False)
        print(f"\nCreated new file. Total records: {len(df_market_sentiment)}")

    print(f"Data saved to: {full_filepath}") # Print the full file path
    print("\nPreview of the latest data:")
    print(df_market_sentiment[['Source', 'Sentiment', 'Text']].head())
else:
    print("No data captured.")

Fetching from: https://economictimes.indiatimes.com/markets/rssfeeds/1977021501.cms
Fetching from: https://www.livemint.com/rss/markets
Fetching from: https://www.business-standard.com/rss/markets-101.rss
Fetching from: https://www.moneycontrol.com/rss/marketreports.xml
Fetching from: https://www.thehindubusinessline.com/markets/feeder/default.rss
Fetching from: https://www.financialexpress.com/market/feed/
Fetching from: https://indianexpress.com/section/business/feed/

Created new file. Total records: 242
Data saved to: /content/drive/MyDrive/MTech_VNIT_3B/Sem3/NLP/RSS/DailyRun/indian_market_RSSFeed_daily_20260501_155906.csv

Preview of the latest data:
                       Source Sentiment  \
145         indianexpress.com  Negative   
89   thehindubusinessline.com  Negative   
85   thehindubusinessline.com   Neutral   
50               livemint.com  Positive   
86   thehindubusinessline.com  Negative   

                                                  Text  
145  ‘Not worried at

In [4]:
nltk.download('maxent_ne_chunker', quiet=True)
nltk.download('words', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True) # Explicitly download the English version
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('maxent_ne_chunker_tab', quiet=True)

def extract_companies(text):
    companies = []
    # Tokenize and Tag parts of speech
    for chunk in nltk.ne_chunk(nltk.pos_tag(nltk.word_tokenize(text))):
        # NLTK labels organizations as 'ORGANIZATION' or 'FACILITY'
        if hasattr(chunk, 'label') and chunk.label() == 'ORGANIZATION':
            companies.append(' '.join(c[0] for c in chunk))

    # Return comma-separated list or NA
    return ", ".join(set(companies)) if companies else "NA"

In [5]:
import pandas as pd
import os

# The target_folder variable is already defined from previous cells.
# It points to '/content/drive/MyDrive/MTech_VNIT_3B/Sem3/NLP/RSS/DailyRun'
# We will use this as both source and destination directory.
source_dir = target_folder
dest_dir = target_folder

# Ensure destination directory exists (though it should already from previous steps)
os.makedirs(dest_dir, exist_ok=True)

all_files = []

# List all CSV files in the source directory
for root, dirs, files in os.walk(source_dir):
    for file in files:
        if file.endswith('.csv'):
            all_files.append(os.path.join(root, file))

if not all_files:
    print(f"No CSV files found in {source_dir}")
else:
    print(f"Found {len(all_files)} CSV files to consolidate in {source_dir}.")
    df_list = []

    for f in all_files:
        try:
            df = pd.read_csv(f)
            df_list.append(df)
        except Exception as e:
            print(f"Error reading {f}: {e}")

    if df_list:
        consolidated_df = pd.concat(df_list, ignore_index=True)

        # --- STEP 1: Save consolidated data with a '_con' suffix ---
        output_filename_con = 'consolidated_DailyRun_data_con.csv'
        output_path_con = os.path.join(dest_dir, output_filename_con)
        consolidated_df.to_csv(output_path_con, index=False)
        print(f"\nSuccessfully consolidated {len(consolidated_df)} rows to: {output_path_con}")
        print("Preview of consolidated data (without duplicate marking):")
        print(consolidated_df.head())

        # --- STEP 2: Mark duplicates and save with a '_filter' suffix ---
        # Create a copy for duplicate marking to ensure the _con file is unchanged
        df_with_duplicates_marked = consolidated_df.copy()

        if 'Text' in df_with_duplicates_marked.columns:
            df_with_duplicates_marked['Duplicate_Row_Number'] = None
            duplicate_mask = df_with_duplicates_marked.duplicated(subset=['Text'], keep=False)
            duplicate_rows = df_with_duplicates_marked[duplicate_mask]

            for text_value in duplicate_rows['Text'].unique():
                indices = df_with_duplicates_marked[df_with_duplicates_marked['Text'] == text_value].index.tolist()
                if len(indices) > 1:
                    first_occurrence_index = indices[0]
                    df_with_duplicates_marked.loc[indices, 'Duplicate_Row_Number'] = first_occurrence_index
            print("\nDuplicate 'Text' entries marked with their first occurrence's row number.")
        else:
            print("\nWarning: 'Text' column not found in consolidated data. Skipping duplicate marking for _filter file.")

        output_filename_filter = 'consolidated_DailyRun_data_filter.csv'
        output_path_filter = os.path.join(dest_dir, output_filename_filter)
        df_with_duplicates_marked.to_csv(output_path_filter, index=False)
        print(f"Successfully saved data with duplicates marked to: {output_path_filter}")
        print("Preview of data with duplicates marked:")
        print(df_with_duplicates_marked.head())

    else:
        print("No dataframes were successfully loaded for consolidation.")

Found 6 CSV files to consolidate in /content/drive/MyDrive/MTech_VNIT_3B/Sem3/NLP/RSS/DailyRun.

Successfully consolidated 1417 rows to: /content/drive/MyDrive/MTech_VNIT_3B/Sem3/NLP/RSS/DailyRun/consolidated_DailyRun_data_con.csv
Preview of consolidated data (without duplicate marking):
                         Source                 Date  \
0  economictimes.indiatimes.com  2026-04-24 12:54:56   
1      thehindubusinessline.com  2026-04-24 12:50:03   
2                  livemint.com  2026-04-24 12:49:15   
3      thehindubusinessline.com  2026-04-24 12:40:15   
4  economictimes.indiatimes.com  2026-04-24 12:37:21   

                                                Text Sentiment   Score  \
0  Sebi clears 4 IPOs including Yatayat Corporati...  Positive  0.5574   
1  The great un-bundling: How maturing Indian mar...  Positive  0.6249   
2  Small-cap stock under  ₹50 Aayush Wellness jum...  Positive  0.9493   
3  IT rout and crude surge drag Nifty to third st...  Negative -0.1531   
4  B

In [ ]:
# # Read the previously saved CSV file
# df_processed = pd.read_csv(full_filepath)

# # Apply the extract_companies function to the loaded DataFrame
# # The 'extract_companies' function is already defined in the cell above this one.
# if not df_processed.empty:
#     df_processed['Company/s'] = df_processed['Text'].apply(extract_companies)

# # Create the new filename with '_org' appended
# base_name, ext = os.path.splitext(full_filepath)
# new_full_filepath = f"{base_name}_org{ext}"

# # Save the DataFrame with the new 'Company/s' column to the new CSV file
# df_processed.to_csv(new_full_filepath, index=False)

# print(f"Processed data with company names saved to: {new_full_filepath}")
# print("\nPreview of the data with extracted companies:")
# print(df_processed[['Source', 'Sentiment', 'Company/s', 'Text']].head())

Processed data with company names saved to: /content/drive/MyDrive/MTech_VNIT_3B/Sem3/NLP/RSS/DailyRun/indian_market_RSSFeed_daily_20260430_153251_org.csv

Preview of the data with extracted companies:
                         Source Sentiment                        Company/s  \
0      thehindubusinessline.com   Neutral                              MSP   
1                  livemint.com  Positive  GMP, OFS, OnEMI Technology, IPO   
2      thehindubusinessline.com  Negative                              HUL   
3  economictimes.indiatimes.com  Positive             Federal Reserve, Fed   
4      thehindubusinessline.com  Positive                              SIF   

                                                Text  
0  Rajasthan farmers appeal Centre to buy 100% ch...  
1  OnEMI Technology IPO: Issue receives tepid res...  
2  Q4 results highlights today: Adani Enterprises...  
3  Kevin Warsh wanted a family fight at the Fed. ...  
4  Union MF launches first long-short equity SIF ...  

In [ ]:
# import pandas as pd
# import os

# # Define the source and destination directories
# source_dir = '/content/drive/MyDrive/MTech_VNIT_3B/Sem3/NLP/Prj assignment /dump'
# dest_dir = '/content/drive/MyDrive/MTech_VNIT_3B/Sem3/NLP/RSS'

# # Ensure destination directory exists
# os.makedirs(dest_dir, exist_ok=True)

# all_files = []

# # List all CSV files in the source directory
# for root, dirs, files in os.walk(source_dir):
#     for file in files:
#         if file.endswith('.csv'):
#             all_files.append(os.path.join(root, file))

# if not all_files:
#     print(f"No CSV files found in {source_dir}")
# else:
#     print(f"Found {len(all_files)} CSV files to consolidate.")
#     df_list = []

#     for f in all_files:
#         try:
#             df = pd.read_csv(f)
#             df_list.append(df)
#         except Exception as e:
#             print(f"Error reading {f}: {e}")

#     if df_list:
#         consolidated_df = pd.concat(df_list, ignore_index=True)

#         # --- STEP 1: Save consolidated data with '_con' suffix ---
#         output_filename_con = 'consolidated_data_con.csv'
#         output_path_con = os.path.join(dest_dir, output_filename_con)
#         consolidated_df.to_csv(output_path_con, index=False)
#         print(f"\nSuccessfully consolidated {len(consolidated_df)} rows to: {output_path_con}")
#         print("Preview of consolidated data (without duplicate marking):")
#         print(consolidated_df.head())

#         # --- STEP 2: Mark duplicates and save with '_filter' suffix ---
#         # Create a copy for duplicate marking to ensure the _con file is unchanged
#         df_with_duplicates_marked = consolidated_df.copy()

#         if 'Text' in df_with_duplicates_marked.columns:
#             df_with_duplicates_marked['Duplicate_Row_Number'] = None
#             duplicate_mask = df_with_duplicates_marked.duplicated(subset=['Text'], keep=False)
#             duplicate_rows = df_with_duplicates_marked[duplicate_mask]

#             for text_value in duplicate_rows['Text'].unique():
#                 indices = df_with_duplicates_marked[df_with_duplicates_marked['Text'] == text_value].index.tolist()
#                 if len(indices) > 1:
#                     first_occurrence_index = indices[0]
#                     df_with_duplicates_marked.loc[indices, 'Duplicate_Row_Number'] = first_occurrence_index
#             print("\nDuplicate 'Text' entries marked with their first occurrence's row number.")
#         else:
#             print("\nWarning: 'Text' column not found in consolidated data. Skipping duplicate marking for _filter file.")

#         output_filename_filter = 'consolidated_data_filter.csv'
#         output_path_filter = os.path.join(dest_dir, output_filename_filter)
#         df_with_duplicates_marked.to_csv(output_path_filter, index=False)
#         print(f"Successfully saved data with duplicates marked to: {output_path_filter}")
#         print("Preview of data with duplicates marked:")
#         print(df_with_duplicates_marked.head())

#     else:
#         print("No dataframes were successfully loaded for consolidation.")

Found 7 CSV files to consolidate.

Successfully consolidated 1273 rows to: /content/drive/MyDrive/MTech_VNIT_3B/Sem3/NLP/RSS/consolidated_data_con.csv
Preview of consolidated data (without duplicate marking):
         Date                                               Text Sentiment  \
0  2026-04-03  Sobha Q4 biz update: Sales rise 11% YoY to Rs ...  Positive   
1  2026-04-03  Quote of the day by Warren Buffett: “Be fearfu...  Negative   
2  2026-04-03  HDFC Securities sees muted Q4 for IT pack; pic...  Positive   
3  2026-04-03  Holiday Halt: Why US stock market is closed fo...  Positive   
4  2026-04-03  SWP planning: 5 conservative hybrid funds that...   Neutral   

    Score Source  
0  0.2960    NaN  
1 -0.8519    NaN  
2  0.8807    NaN  
3  0.8074    NaN  
4  0.0000    NaN  

Duplicate 'Text' entries marked with their first occurrence's row number.
Successfully saved data with duplicates marked to: /content/drive/MyDrive/MTech_VNIT_3B/Sem3/NLP/RSS/consolidated_data_filter.csv
Pre